<a href="https://colab.research.google.com/github/officiallong/Fork-Test-Run/blob/master/Exercise_1_Prompt_Chaining_Long_Nguyen.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **Exercise 1 — Prompt Chaining for Customer Support AI**

# **Tools Used:**
1. Google Colab
2. Python
3. Google Gemini API
4. google-genai Python SDK
5. Gemini model

# **Purpose**
For this exercise, I created a customer support AI workflow using prompt chaining. The goal was to take a customer issue through multiple steps, where the output from one prompt was used as input for the next prompt.

In [8]:
!pip install -q -U google-genai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 262.4/262.4 kB 14.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.58.0 which is incompatible.


# **Prompt 1 — Issue Classification**

In [10]:
from google import genai
from google.colab import userdata

client = genai.Client(api_key=userdata.get("GEMINI_API_KEY"))

print("Gemini client connected successfully.")

Gemini client connected successfully.


In [11]:
def ask_ai(prompt):
    response = client.models.generate_content(
        model="gemini-3.8-flash",
        contents=prompt
    )
    return response.text

In [12]:
customer_message = """
I was charged twice for my order. I only placed one order
and I need help getting the extra charge refunded.
"""

print("Customer Message:")
print(customer_message)

Customer Message:

I was charged twice for my order. I only placed one order
and I need help getting the extra charge refunded.



In [13]:
prompt_1 = f"""
You are a customer support AI.

Classify the following customer issue into exactly one category:
Billing, Technical, Account, Order, or Other.

Customer message:
{customer_message}

Return only the category.
"""

category = ask_ai(prompt_1)

print("STEP 1 - ISSUE CLASSIFICATION")
print(category)

STEP 1 - ISSUE CLASSIFICATION
Billing


# **Prompt 2 — Identify Missing Information**

In [14]:
prompt_2 = f"""
You are a customer support AI.

Customer message:
{customer_message}

The issue was classified in the previous step as:
{category}

Based on the customer's message and the classification,
identify the information still needed to investigate the issue.

Constraints:
- Do not invent customer or account information.
- Ask only for information relevant to resolving the issue.
- Keep the response concise.

Return a short list of the missing information.
"""

missing_info = ask_ai(prompt_2)

print("STEP 2 - MISSING INFORMATION")
print(missing_info)

STEP 2 - MISSING INFORMATION
To investigate and resolve this issue, the following information is needed:

* **Order number** (or confirmation number)
* **Email address** or name associated with the account/order
* **Transaction details** (exact amount(s) charged, date of the charges, and the last 4 digits of the payment method used)


# **Prompt 3 — Proposed Solution**

In [16]:
prompt_3 = f"""
You are a professional customer support AI.

Customer message:
{customer_message}

Issue category from Step 1:
{category}

Missing information identified in Step 2:
{missing_info}

Based on all of the information above, recommend an appropriate next step
for resolving the customer's issue.

Constraints:
- Use a professional and helpful tone.
- Do not claim that a refund has already been issued.
- Do not invent customer or account information.
- Keep the response concise.

Return a short proposed solution.
"""

solution = ask_ai(prompt_3)

print("STEP 3 - PROPOSED SOLUTION")
print(solution)

STEP 3 - PROPOSED SOLUTION
Thank you for reaching out, and I apologize for the inconvenience caused by the duplicate charge. I would be glad to help resolve this for you. 

To help us investigate the charges and process a refund for the duplicate payment, please reply with the following details:

* Your **order number** (or confirmation number)
* The **email address** associated with the order
* The **exact amount(s)**, **date of the charges**, and the **last 4 digits** of the payment card used

Once we have this information, we will review the transactions immediately and assist you with the resolution.


# **Prompt 4 — Escalation Decision**

In [17]:
prompt_4 = f"""
You are a customer support AI responsible for deciding
whether a case requires human assistance.

Customer message:
{customer_message}

Issue category from Step 1:
{category}

Missing information from Step 2:
{missing_info}

Proposed solution from Step 3:
{solution}

Based on the complete case, determine whether this issue should be
escalated to a human customer support representative.

Escalate if:
- The issue requires access to payment or account records.
- A refund or financial adjustment may need to be processed.
- The AI cannot safely complete the resolution on its own.

Return exactly this format:

Decision: Escalate or Do Not Escalate
Reason: One brief explanation
"""

escalation = ask_ai(prompt_4)

print("STEP 4 - ESCALATION DECISION")
print(escalation)

STEP 4 - ESCALATION DECISION
Decision: Escalate
Reason: Resolving a duplicate charge requires accessing payment records and processing a financial refund, which must be handled safely by a human representative.


In [18]:
test_message = """
My account is locked and I cannot log in.
I already tried resetting my password twice, but it still does not work.
"""

print("TEST CUSTOMER MESSAGE:")
print(test_message)

TEST CUSTOMER MESSAGE:

My account is locked and I cannot log in.
I already tried resetting my password twice, but it still does not work.



In [20]:
test_prompt_1 = f"""
You are a customer support AI.

Classify the following customer issue into exactly one category:
Billing, Technical, Account, Order, or Other.

Customer message:
{test_message}

Return only the category.
"""

test_category = ask_ai(test_prompt_1)

print("TEST - ORIGINAL PROMPT")
print("Classification:", test_category)

TEST - ORIGINAL PROMPT
Classification: Account


In [21]:
improved_test_prompt = f"""
You are a customer support AI.

Classify the following customer issue into exactly one category:
Billing, Technical, Account, Order, or Other.

Customer message:
{test_message}

Instructions:
- Choose the single category that best represents the main issue.
- Base the classification only on information in the customer's message.
- Do not invent any details.
- If multiple categories could apply, choose the category most directly
  related to the customer's primary problem.
- Provide a brief reason for the classification.

Return exactly this format:
Category: [category]
Reason: [one brief sentence]
"""

improved_test_category = ask_ai(improved_test_prompt)

print("TEST - IMPROVED PROMPT")
print(improved_test_category)

TEST - IMPROVED PROMPT
Category: Account
Reason: The customer is unable to access their profile because their account is locked and password resets have failed.


**Test and Iteration:**

I tested the original classification prompt with a second customer issue about a locked account. The original prompt correctly classified the issue as "Account," but it only returned the category and did not explain why it made that decision.

To improve the prompt, I added clearer instructions for choosing the main issue, told the model not to make assumptions beyond the customer’s message, and asked it to provide a short reason for its classification. After testing the updated prompt, it still classified the issue as "Account," but it also explained that the customer could not access their account because it was locked and the password resets had failed.

The updated prompt made the result easier to understand because it showed both the classification and the reasoning behind it while keeping the response concise.